# TumorHeal Advanced Analytics Examples

This notebook demonstrates the usage of TumorHeal's advanced analytics features including:
- Survival Analysis
- Dimensionality Reduction
- Real-time Analytics
- Time-to-Event Prediction

## Setup
First, we'll install and import all required packages.

In [ ]:
# Install required packages
!pip install pandas numpy scipy scikit-learn tensorflow lifelines umap-learn

# Import packages
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from lifelines import KaplanMeierFitter, CoxPHFitter
import tensorflow as tf
from datetime import datetime, timedelta
import logging

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

## 1. Generate Sample Data

Let's generate some sample data to demonstrate each analytics feature:
1. Survival data for survival analysis
2. High-dimensional data for dimensionality reduction
3. Time series data for real-time analytics
4. Event data for time-to-event prediction

In [ ]:
# Generate survival data
np.random.seed(42)
n_patients = 1000

survival_data = pd.DataFrame({
    'duration': np.random.exponential(50, n_patients),
    'event': np.random.binomial(1, 0.7, n_patients),
    'age': np.random.normal(60, 10, n_patients),
    'treatment': np.random.choice(['A', 'B', 'C'], n_patients),
    'stage': np.random.choice(['I', 'II', 'III'], n_patients)
})

# Generate high-dimensional data
n_features = 50
n_samples = 500

high_dim_data = pd.DataFrame(
    np.random.normal(0, 1, (n_samples, n_features)),
    columns=[f'feature_{i}' for i in range(n_features)]
)

# Add some structure to the data
high_dim_data['class'] = np.random.choice(['A', 'B', 'C'], n_samples)

# Generate time series data
dates = pd.date_range(
    start='2023-01-01',
    end='2023-01-02',
    freq='1min'
)

n_timepoints = len(dates)
time_series_data = pd.DataFrame({
    'temperature': np.sin(np.linspace(0, 8*np.pi, n_timepoints)) + \
                  np.random.normal(0, 0.1, n_timepoints),
    'pressure': np.cos(np.linspace(0, 8*np.pi, n_timepoints)) + \
               np.random.normal(0, 0.1, n_timepoints),
    'flow_rate': np.exp(np.linspace(0, 2, n_timepoints)) + \
                np.random.normal(0, 0.5, n_timepoints)
}, index=dates)

# Generate event data
n_events = 500
event_data = pd.DataFrame({
    'time_to_event': np.random.exponential(30, n_events),
    'biomarker_1': np.random.normal(100, 15, n_events),
    'biomarker_2': np.random.normal(5, 1, n_events),
    'age': np.random.normal(60, 10, n_events),
    'treatment_duration': np.random.uniform(10, 90, n_events)
})

## 2. Survival Analysis

Let's implement survival analysis using both Kaplan-Meier and Cox Proportional Hazards models:

In [ ]:
async def survival_analysis(data, duration_col, event_col, groups=None):
    """Perform survival analysis"""
    # Initialize KM fitter
    kmf = KaplanMeierFitter()
    
    # Analyze by groups if specified
    if groups:
        results = {}
        for group in data[groups].unique():
            mask = data[groups] == group
            kmf.fit(
                data.loc[mask, duration_col],
                data.loc[mask, event_col],
                label=str(group)
            )
            results[str(group)] = {
                'survival_function': kmf.survival_function_.to_dict(),
                'median_survival': kmf.median_survival_time_,
                'confidence_intervals': kmf.confidence_interval_.to_dict()
            }
    else:
        kmf.fit(data[duration_col], data[event_col])
        results = {
            'all': {
                'survival_function': kmf.survival_function_.to_dict(),
                'median_survival': kmf.median_survival_time_,
                'confidence_intervals': kmf.confidence_interval_.to_dict()
            }
        }
    
    # Fit Cox model
    cph = CoxPHFitter()
    covariates = [col for col in data.columns 
                 if col not in [duration_col, event_col, groups]]
    
    if covariates:
        cph.fit(data, duration_col=duration_col, event_col=event_col)
        cox_results = {
            'hazard_ratios': cph.hazard_ratios_.to_dict(),
            'confidence_intervals': cph.confidence_intervals_.to_dict(),
            'p_values': cph.print_summary().loc[:, 'p'].to_dict()
        }
    else:
        cox_results = None
        
    return results, cox_results

# Analyze survival by treatment group
results, cox_results = await survival_analysis(
    survival_data,
    'duration',
    'event',
    'treatment'
)

# Print results
print("Survival Analysis Results:")
print("\nKaplan-Meier Results:")
for group, stats in results.items():
    print(f"\nGroup: {group}")
    print(f"Median survival time: {stats['median_survival']:.2f}")
    
print("\nCox Model Results:")
if cox_results:
    print("\nHazard Ratios:")
    for var, hr in cox_results['hazard_ratios'].items():
        print(f"{var}: {hr:.2f}")

## 3. Dimensionality Reduction

Now let's reduce the dimensionality of our high-dimensional data using different methods:
1. t-SNE
2. UMAP
3. PCA

In [ ]:
async def reduce_dimensions(data, n_components=2, method='tsne'):
    """Reduce data dimensionality"""
    # Scale data
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(data)
    
    # Reduce dimensions
    if method == 'tsne':
        tsne = TSNE(n_components=n_components, random_state=42)
        reduced_data = tsne.fit_transform(scaled_data)
    elif method == 'umap':
        import umap
        reducer = umap.UMAP(n_components=n_components)
        reduced_data = reducer.fit_transform(scaled_data)
    else:  # PCA
        pca = PCA(n_components=n_components)
        reduced_data = pca.fit_transform(scaled_data)
        explained_variance = pca.explained_variance_ratio_
        print(f"\nExplained variance ratios: {explained_variance}")
        
    return reduced_data

# Prepare feature data
feature_cols = [col for col in high_dim_data.columns if col != 'class']
X = high_dim_data[feature_cols]

# Apply different reduction methods
methods = ['tsne', 'umap', 'pca']
results = {}

for method in methods:
    results[method] = await reduce_dimensions(X, method=method)
    print(f"\n{method.upper()} Results:")
    print(f"Reduced data shape: {results[method].shape}")

# Plot results
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Dimensionality Reduction Comparison')

for i, (method, reduced_data) in enumerate(results.items()):
    axes[i].scatter(
        reduced_data[:, 0],
        reduced_data[:, 1],
        c=pd.Categorical(high_dim_data['class']).codes,
        cmap='viridis'
    )
    axes[i].set_title(method.upper())
    axes[i].set_xlabel('Component 1')
    axes[i].set_ylabel('Component 2')

plt.tight_layout()
plt.show()

## 4. Real-time Analytics

Let's analyze real-time streaming data with rolling statistics and anomaly detection:

In [ ]:
async def analyze_real_time(data, window_size='10min'):
    """Analyze real-time streaming metrics"""
    # Calculate rolling statistics
    rolling = data.rolling(window=window_size)
    
    results = {
        'current_values': data.iloc[-1].to_dict(),
        'rolling_mean': rolling.mean().iloc[-1].to_dict(),
        'rolling_std': rolling.std().iloc[-1].to_dict(),
        'rolling_min': rolling.min().iloc[-1].to_dict(),
        'rolling_max': rolling.max().iloc[-1].to_dict(),
        'velocity': data.diff().iloc[-1].to_dict(),
        'acceleration': data.diff().diff().iloc[-1].to_dict()
    }
    
    # Detect anomalies
    anomalies = {}
    for column in data.columns:
        values = data[column]
        rolling_mean = rolling[column].mean()
        rolling_std = rolling[column].std()
        
        # Z-score based anomalies
        z_scores = abs((values - rolling_mean) / rolling_std)
        anomalies[column] = {
            'timestamp': data.index[-1],
            'value': float(values.iloc[-1]),
            'z_score': float(z_scores.iloc[-1]),
            'is_anomaly': bool(z_scores.iloc[-1] > 3)
        }
    
    return results, anomalies

# Analyze real-time data
results, anomalies = await analyze_real_time(
    time_series_data,
    window_size='10min'
)

# Print results
print("Real-time Analytics Results:")
print("\nCurrent Values:")
for metric, value in results['current_values'].items():
    print(f"{metric}: {value:.2f}")

print("\nRolling Statistics:")
print("Mean:")
for metric, value in results['rolling_mean'].items():
    print(f"{metric}: {value:.2f}")

print("\nAnomalies Detected:")
for metric, anomaly in anomalies.items():
    if anomaly['is_anomaly']:
        print(f"{metric}: Z-score = {anomaly['z_score']:.2f}")

# Plot time series with anomalies
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 1, figsize=(15, 10))
fig.suptitle('Time Series Analysis with Anomaly Detection')

for i, column in enumerate(time_series_data.columns):
    axes[i].plot(time_series_data.index, time_series_data[column], label=column)
    
    # Mark anomalies
    if anomalies[column]['is_anomaly']:
        axes[i].scatter(
            anomalies[column]['timestamp'],
            anomalies[column]['value'],
            color='red',
            marker='o',
            s=100,
            label='Anomaly'
        )
    
    axes[i].set_title(f"{column} Time Series")
    axes[i].legend()
    axes[i].grid(True)

plt.tight_layout()
plt.show()

## 5. Time-to-Event Prediction

Finally, let's implement time-to-event prediction using neural networks:

In [ ]:
async def predict_time_to_event(data, target_col, features, threshold):
    """Predict time until specific events occur"""
    # Prepare features
    X = data[features]
    y = data[target_col]
    
    # Scale features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Split data
    from sklearn.model_selection import train_test_split
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, random_state=42
    )
    
    # Build model
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(1)
    ])
    
    model.compile(optimizer='adam', loss='mse')
    
    # Train model
    history = model.fit(
        X_train,
        y_train,
        epochs=50,
        validation_split=0.2,
        verbose=0
    )
    
    # Make predictions
    predictions = model.predict(X_scaled)
    
    # Calculate prediction intervals
    errors = predictions.flatten() - y
    std_error = np.std(errors)
    
    intervals = {
        'lower': predictions.flatten() - 1.96 * std_error,
        'upper': predictions.flatten() + 1.96 * std_error
    }
    
    # Calculate event probabilities
    time_windows = [1, 7, 30, 90]  # days
    probabilities = {}
    
    for window in time_windows:
        mask = y <= window
        prob = np.mean(mask)
        probabilities[f'{window}d'] = float(prob)
    
    return predictions, intervals, probabilities, history.history

# Prepare data for prediction
features = ['biomarker_1', 'biomarker_2', 'age']
predictions, intervals, probabilities, history = await predict_time_to_event(
    event_data,
    'time_to_event',
    features,
    threshold=30
)

# Plot results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Time-to-Event Prediction Results')

# Plot predictions vs actual
ax1.scatter(event_data['time_to_event'], predictions, alpha=0.5)
ax1.plot([0, max(event_data['time_to_event'])],
         [0, max(event_data['time_to_event'])],
         'r--', label='Perfect Prediction')
ax1.set_xlabel('Actual Time')
ax1.set_ylabel('Predicted Time')
ax1.set_title('Predictions vs Actual')
ax1.legend()

# Plot training history
ax2.plot(history['loss'], label='Training Loss')
ax2.plot(history['val_loss'], label='Validation Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.set_title('Training History')
ax2.legend()

plt.tight_layout()
plt.show()

# Print probabilities
print("\nEvent Probabilities:")
for window, prob in probabilities.items():
    print(f"Probability of event within {window}: {prob:.2%}")